In [8]:
import os
os.makedirs('../outputs/figures', exist_ok=True)

In [10]:
!pip install contextily

In [6]:
import geemap
import ee
from ipyleaflet import Marker, DivIcon, WidgetControl
from ipywidgets import HTML
import matplotlib.patheffects as pe

ee.Initialize()

aoi_coords = [
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]

stations = {
    'Isla Boquerón': (-74.298, 10.962, 'INVEMAR'),
    'Punta Cerro': (-74.283, 10.973, 'INVEMAR'),
    'Punta Chino': (-74.305, 10.912, 'INVEMAR'),
    'Río Sevilla': (-74.325, 10.880, 'INVEMAR'),
    'Caño Palos': (-74.471, 10.758, 'INVEMAR'),
    'CP Pajarales': (-74.75, 10.85, 'Complementaria'),
    'Caño Clarín': (-74.55, 10.55, 'Complementaria'),
    'VIPIS': (-74.65, 11.02, 'Complementaria'),
}

Map = geemap.Map(center=[10.72, -74.45], zoom=10, layout={'height': '700px'})
Map.add_basemap('SATELLITE')

# Composite Sentinel-2 2024 como capa adicional
def mask_s2(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

s2_rgb = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(ee.Geometry.Polygon(aoi_coords))
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2)
    .median()
    .select(['B4', 'B3', 'B2']))

Map.addLayer(s2_rgb, {'min': 0, 'max': 3000}, 'Sentinel-2 RGB 2024', opacity=0.9)

# AOI
aoi = ee.Geometry.Polygon(aoi_coords)
styled_aoi = ee.FeatureCollection([ee.Feature(aoi)]).style(
    color='FF3333', fillColor='00000000', width=3)
Map.addLayer(styled_aoi, {}, 'Área de estudio')

# Estaciones INVEMAR (círculos rojos)
invemar_pts = []
comple_pts = []
for name, (lon, lat, tipo) in stations.items():
    pt = ee.Feature(ee.Geometry.Point([lon, lat]).buffer(600), {'name': name})
    if tipo == 'INVEMAR':
        invemar_pts.append(pt)
    else:
        comple_pts.append(pt)

Map.addLayer(
    ee.FeatureCollection(invemar_pts).style(color='FF3333', fillColor='FF333399', width=2),
    {}, 'Estaciones INVEMAR')
Map.addLayer(
    ee.FeatureCollection(comple_pts).style(color='FFB300', fillColor='FFB30099', width=2),
    {}, 'Estaciones complementarias')

# Labels de estaciones (sin fondo)
for name, (lon, lat, tipo) in stations.items():
    color = '#FF4444' if tipo == 'INVEMAR' else '#FFB300'
    icon = DivIcon(
        html=f'<div style="font-size:11px; font-weight:bold; color:{color}; '
             f'text-shadow:2px 2px 4px black, -1px -1px 3px black, 1px -1px 3px black, -1px 1px 3px black; '
             f'white-space:nowrap; background:none !important; border:none !important; '
             f'box-shadow:none !important;">{name}</div>',
        icon_size=[0, 0],
        icon_anchor=[-10, 25],
        class_name=''
    )
    Map.add(Marker(location=[lat, lon], icon=icon, draggable=False))

# Labels geográficos (sin fondo)
geo_labels = {
    'Ciénaga Grande de Santa Marta': (10.84, -74.45, '#7EC8E3', 14),
    'Mar Caribe': (11.07, -74.50, '#A8D8EA', 16),
    'Complejo de Pajarales': (10.80, -74.74, '#A8E6A1', 11),
    'Isla de Salamanca': (11.035, -74.65, '#A8E6A1', 11),
}
for label, (lat, lon, color, size) in geo_labels.items():
    icon = DivIcon(
        html=f'<div style="font-size:{size}px; font-style:italic; color:{color}; '
             f'text-shadow:2px 2px 4px black, -1px -1px 3px black, 1px -1px 3px black, -1px 1px 3px black; '
             f'white-space:nowrap; background:none !important; border:none !important; '
             f'box-shadow:none !important;">{label}</div>',
        icon_size=[0, 0],
        icon_anchor=[60, 10],
        class_name=''
    )
    Map.add(Marker(location=[lat, lon], icon=icon, draggable=False))

# Leyenda HTML personalizada
legend_html = """
<div style="
    background: rgba(255,255,255,0.92); 
    padding: 10px 14px; 
    border-radius: 6px; 
    border: 1px solid #999;
    font-family: Arial, sans-serif;
    font-size: 12px;
    line-height: 24px;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
    <b style="font-size:13px; color:#1a1a1a;">Leyenda</b><br>
    <svg width="14" height="14" style="vertical-align:middle;">
        <circle cx="7" cy="7" r="6" fill="#FF3333" stroke="white" stroke-width="1.5"/>
    </svg> <span style="color:#1a1a1a;">Estaciones INVEMAR (5)</span><br>
    <svg width="14" height="14" style="vertical-align:middle;">
        <circle cx="7" cy="7" r="6" fill="#FFB300" stroke="white" stroke-width="1.5"/>
    </svg> <span style="color:#1a1a1a;">Estaciones complementarias (3)</span><br>
    <svg width="16" height="14" style="vertical-align:middle;">
        <line x1="0" y1="7" x2="16" y2="7" stroke="#FF3333" stroke-width="3"/>
    </svg> <span style="color:#1a1a1a;">Área de estudio (5.073 km²)</span>
</div>
"""
Map.add(WidgetControl(widget=HTML(value=legend_html), position='bottomleft'))
# Grilla de coordenadas
Map.add_layer_control()

# Barra de escala (geemap la tiene incorporada)
from ipyleaflet import ScaleControl
Map.add(ScaleControl(position='bottomright'))

# Flecha norte como HTML
north_html = """
<div style="
    background: rgba(0,0,0,0.6);
    padding: 6px 10px;
    border-radius: 4px;
    text-align: center;
    line-height: 1.2;
">
    <span style="color:white; font-size:14px; font-weight:bold;">N</span><br>
    <span style="color:white; font-size:20px;">▲</span>
</div>
"""
Map.add(WidgetControl(widget=HTML(value=north_html), position='topright'))

# Gratícula cada 0.2 grados
import ee
graticule_lines = []
for lon in [-74.8, -74.6, -74.4, -74.2]:
    graticule_lines.append(ee.Feature(ee.Geometry.LineString([[lon, 10.3], [lon, 11.1]])))
for lat in [10.4, 10.6, 10.8, 11.0]:
    graticule_lines.append(ee.Feature(ee.Geometry.LineString([[-74.9, lat], [-74.1, lat]])))

graticule_fc = ee.FeatureCollection(graticule_lines).style(
    color='FFFFFF40', width=1)
Map.addLayer(graticule_fc, {}, 'Gratícula')

Map

Map(center=[10.72, -74.45], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…